# scene3d — Colab Pipeline: MASt3R-SLAM + gsplat + Grounded-SAM-2

Runs all GPU-heavy stages on Google Colab. Your Mac handles preprocessing and the viewer.

**Before running:** Runtime → Change runtime type → T4 GPU (free) or A100 (Pro)

## Workflow
1. Upload preprocessed frames to Google Drive (`scene3d/{SCENE_NAME}/frames/`)
2. Run this notebook top-to-bottom
3. Download results to Mac from Google Drive (`scene3d/{SCENE_NAME}/outputs/`)

---
## 0. Mount Google Drive & Configure

In [ ]:
# FIX: drive.mount was missing entirely from the old notebook.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# FIX: this entire block was inside a Markdown cell in the old notebook.
# Markdown cells never execute — all path variables were undefined, causing
# NameError on every subsequent cell.
import os, shutil, glob, struct, json, pathlib
import numpy as np
import cv2

# ── EDIT THIS ────────────────────────────────────────────────
SCENE_NAME = "scene1"   # change per scene
# ─────────────────────────────────────────────────────────────

DRIVE_FRAMES  = f"/content/drive/MyDrive/scene3d/{SCENE_NAME}/frames"
DRIVE_OUTPUT  = f"/content/drive/MyDrive/scene3d/{SCENE_NAME}/outputs"

WORK_DIR      = f"/content/scene3d_work/{SCENE_NAME}"
FRAMES_DIR    = f"{WORK_DIR}/frames"
SLAM_DIR      = "/content/MASt3R-SLAM"
SLAM_LOGS     = f"{SLAM_DIR}/logs"
COLMAP_DIR    = f"{WORK_DIR}/colmap"
GSPLAT_OUTPUT = f"{WORK_DIR}/gsplat_output"
GSPLAT_REPO   = "/content/gsplat"
MASKS_DIR     = f"{WORK_DIR}/masks"

# Runtime globals (set later)
SLAM_TRAJ = None
SLAM_PLY  = None
FINAL_PLY = None

os.makedirs(WORK_DIR, exist_ok=True)
print(f"Scene : {SCENE_NAME}")
print(f"Frames: {DRIVE_FRAMES}")
print(f"Output: {DRIVE_OUTPUT}")

In [ ]:
# Copy frames from Drive → Colab local SSD (much faster for GPU reads)
if os.path.exists(FRAMES_DIR):
    shutil.rmtree(FRAMES_DIR)
shutil.copytree(DRIVE_FRAMES, FRAMES_DIR)

num_frames = len([f for f in os.listdir(FRAMES_DIR) if f.endswith(('.jpg', '.png'))])
print(f"✓ Copied {num_frames} frames to {FRAMES_DIR}")

In [ ]:
# MASt3R-SLAM's RGBFiles dataloader globs ONLY *.png files.
# Our preprocessor outputs .jpg, so convert before running SLAM.
jpg_files = sorted(glob.glob(os.path.join(FRAMES_DIR, "*.jpg")))
if jpg_files:
    print(f"Converting {len(jpg_files)} .jpg → .png ...")
    for jpg_path in jpg_files:
        png_path = jpg_path.replace('.jpg', '.png')
        img = cv2.imread(jpg_path)
        cv2.imwrite(png_path, img)
        os.remove(jpg_path)
    print(f"✓ Converted {len(jpg_files)} frames")
else:
    png_count = len(glob.glob(os.path.join(FRAMES_DIR, "*.png")))
    print(f"✓ Found {png_count} .png frames (no conversion needed)")

---
## 1. Install MASt3R-SLAM

Takes ~8 minutes on first run. Fixes vs old notebook:
- `TORCH_CUDA_ARCH_LIST` now covers T4 (7.5), A100 (8.0), L4 (8.9), V100 (7.0)
- `MAX_JOBS=2` (not 4) prevents RAM OOM during CUDA kernel compilation
- curope gets `--no-build-isolation --no-deps` to avoid numpy version conflicts
- MASt3R-SLAM gets `--no-deps` to skip `imgui` (GUI, useless headless) and `moderngl-window` (forces numpy downgrade)
- `faiss-cpu`, `trimesh`, `roma` added manually (runtime deps skipped by --no-deps)

In [ ]:
import torch
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA : {torch.version.cuda}")
print(f"Torch: {torch.__version__}")

In [ ]:
if not os.path.exists(SLAM_DIR):
    !git clone --recursive https://github.com/rmurai0610/MASt3R-SLAM.git {SLAM_DIR}
else:
    print("✓ MASt3R-SLAM already cloned")

In [ ]:
# FIX: ARCH_LIST now covers all common Colab GPUs (was "7.5" only = T4 only)
# FIX: MAX_JOBS=2 prevents RAM OOM on T4 (was 4)
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.0;7.5;8.0;8.9"  # V100;T4;A100;L4
os.environ["FORCE_CUDA"] = "1"
os.environ["MAX_JOBS"] = "2"

# Fix build tools (distutils removed in Python 3.12)
!pip install -q setuptools==69.5.1 wheel ninja

# lietorch — SE3/SO3 CUDA bindings
!pip install --no-build-isolation git+https://github.com/princeton-vl/lietorch.git

# FIX: curope now gets --no-build-isolation --no-deps
# (was missing both flags → numpy conflict caused compile failure)
!cd {SLAM_DIR}/thirdparty/mast3r/dust3r/croco/models/curope && \
    pip install --no-build-isolation --no-deps .

# MASt3R backbone (3D matching)
# FIX: --no-deps skips re-triggering curope build with wrong flags
!pip install --no-build-isolation --no-deps -e {SLAM_DIR}/thirdparty/mast3r

# in3d utility library
!pip install --no-build-isolation --no-deps -e {SLAM_DIR}/thirdparty/in3d

# FIX: rm -rf build before install (stale .so from partial builds caused silent failures)
# FIX: --no-deps skips imgui (GUI, useless headless) and moderngl-window (numpy downgrade)
!cd {SLAM_DIR} && rm -rf build && pip install --no-build-isolation --no-deps -e .

# FIX: install the runtime deps that --no-deps skipped
# (faiss-cpu, trimesh, roma were missing entirely from the old notebook)
!pip install -q faiss-cpu trimesh roma einops plyfile natsort

print("\n✓ MASt3R-SLAM installed")

In [ ]:
CKPT_DIR   = f"{SLAM_DIR}/checkpoints"
MASTR_CKPT = f"{CKPT_DIR}/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"
RETR_CKPT  = f"{CKPT_DIR}/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_trainingfree.pth"
os.makedirs(CKPT_DIR, exist_ok=True)

if not os.path.exists(MASTR_CKPT):
    print("Downloading MASt3R checkpoint (~1.3 GB)...")
    !wget -q --show-progress -O {MASTR_CKPT} \
        "https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"
else:
    print("✓ MASt3R checkpoint exists")

if not os.path.exists(RETR_CKPT):
    print("Downloading retrieval checkpoint...")
    !wget -q --show-progress -O {RETR_CKPT} \
        "https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_trainingfree.pth"
else:
    print("✓ Retrieval checkpoint exists")

print(f"  MASt3R checkpoint: {os.path.getsize(MASTR_CKPT)/1e6:.0f} MB")

---
## 2. Run MASt3R-SLAM

CLI flags (verified from `main.py` source):
- `--dataset` : path to image folder (`.png` only) or `.mp4`
- `--config`  : YAML config
- `--no-viz`  : headless mode (required on Colab)
- `--save-as` : subfolder under `logs/` for outputs

Outputs in `logs/`:
```
frames.txt   ← TUM trajectory: timestamp x y z qx qy qz qw (camera-to-world)
frames.ply   ← dense point cloud
```

In [ ]:
# Clean previous logs to avoid stale data
if os.path.exists(SLAM_LOGS):
    shutil.rmtree(SLAM_LOGS)

%cd {SLAM_DIR}

!python main.py \
    --dataset {FRAMES_DIR} \
    --config config/base.yaml \
    --no-viz \
    --save-as default

print("\n✓ MASt3R-SLAM finished")

In [ ]:
# seq_name is derived from the folder name ("frames")
seq_name     = pathlib.Path(FRAMES_DIR).stem
expected_traj = os.path.join(SLAM_LOGS, f"{seq_name}.txt")
expected_ply  = os.path.join(SLAM_LOGS, f"{seq_name}.ply")

print(f"seq_name     : {seq_name}")
print(f"Trajectory   : {expected_traj} → exists: {os.path.exists(expected_traj)}")
print(f"Point cloud  : {expected_ply}  → exists: {os.path.exists(expected_ply)}")

print("\nAll files in logs/:")
if os.path.exists(SLAM_LOGS):
    for root, dirs, files in os.walk(SLAM_LOGS):
        for f in sorted(files):
            path = os.path.join(root, f)
            print(f"  {os.path.relpath(path, SLAM_LOGS)}: {os.path.getsize(path)/1e6:.2f} MB")
else:
    print("  ⚠ No logs/ directory — searching for outputs:")
    !find {SLAM_DIR} -name "*.ply" -o -name "*.txt" 2>/dev/null | head -20

In [ ]:
# Load trajectory (TUM: timestamp x y z qx qy qz qw — camera-to-world)
traj_candidates = [
    expected_traj,
    *glob.glob(os.path.join(SLAM_LOGS, "**/*.txt"), recursive=True),
]
for traj_path in traj_candidates:
    if not os.path.exists(traj_path) or os.path.getsize(traj_path) < 50:
        continue
    try:
        data = np.loadtxt(traj_path)
        if data.ndim == 2 and data.shape[1] == 8:
            SLAM_TRAJ = data
            print(f"✓ Trajectory: {traj_path} ({len(SLAM_TRAJ)} poses)")
            break
    except Exception:
        continue

if SLAM_TRAJ is None:
    print("⚠ No trajectory found — will use identity poses (gsplat will optimise)")

# FIX: only search inside SLAM_LOGS (old code searched all of SLAM_DIR and
# accidentally picked up sample mustard_bottle.ply shipped with MASt3R-SLAM)
ply_candidates = [
    expected_ply,
    *glob.glob(os.path.join(SLAM_LOGS, "**/*.ply"), recursive=True),
]
for ply_path in ply_candidates:
    if os.path.exists(ply_path) and os.path.getsize(ply_path) > 1000:
        SLAM_PLY = ply_path
        print(f"✓ Point cloud : {ply_path} ({os.path.getsize(ply_path)/1e6:.1f} MB)")
        break

if SLAM_PLY is None:
    print("⚠ No point cloud found — COLMAP will have empty points3D.bin")

---
## 3. Convert to COLMAP Format

gsplat's `simple_trainer.py` expects:
```
colmap/
  images/          ← frame images
  sparse/0/
    cameras.bin    ← intrinsics
    images.bin     ← extrinsics (world-to-camera)
    points3D.bin   ← 3D point cloud
```
MASt3R-SLAM outputs camera-to-world poses — we invert them for COLMAP.

In [ ]:
from plyfile import PlyData

os.makedirs(f"{COLMAP_DIR}/sparse/0", exist_ok=True)

# Copy images into COLMAP workspace
colmap_images = f"{COLMAP_DIR}/images"
if os.path.exists(colmap_images):
    shutil.rmtree(colmap_images)
shutil.copytree(FRAMES_DIR, colmap_images)

frame_names = sorted([f for f in os.listdir(FRAMES_DIR) if f.endswith(('.png', '.jpg'))])
first_img   = cv2.imread(os.path.join(FRAMES_DIR, frame_names[0]))
img_h, img_w = first_img.shape[:2]

focal = float(max(img_w, img_h))
cx, cy = img_w / 2.0, img_h / 2.0

cameras_path = f"{COLMAP_DIR}/sparse/0/cameras.bin"
with open(cameras_path, 'wb') as f:
    f.write(struct.pack('<Q', 1))
    f.write(struct.pack('<i', 1))   # camera_id
    f.write(struct.pack('<i', 1))   # model_id = PINHOLE
    f.write(struct.pack('<Q', img_w))
    f.write(struct.pack('<Q', img_h))
    for p in [focal, focal, cx, cy]:
        f.write(struct.pack('<d', p))

print(f"✓ cameras.bin  {img_w}×{img_h}, focal={focal:.0f}")
print(f"  {len(frame_names)} frames")

In [ ]:
def quat_to_rot(qw, qx, qy, qz):
    return np.array([
        [1-2*(qy**2+qz**2), 2*(qx*qy-qz*qw), 2*(qx*qz+qy*qw)],
        [2*(qx*qy+qz*qw),   1-2*(qx**2+qz**2), 2*(qy*qz-qx*qw)],
        [2*(qx*qz-qy*qw),   2*(qy*qz+qx*qw),   1-2*(qx**2+qy**2)],
    ])

def rot_to_quat(R):
    trace = R[0,0] + R[1,1] + R[2,2]
    if trace > 0:
        s = 0.5 / np.sqrt(trace + 1.0)
        return 0.25/s, (R[2,1]-R[1,2])*s, (R[0,2]-R[2,0])*s, (R[1,0]-R[0,1])*s
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s = 2.0 * np.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2])
        return (R[2,1]-R[1,2])/s, 0.25*s, (R[0,1]+R[1,0])/s, (R[0,2]+R[2,0])/s
    elif R[1,1] > R[2,2]:
        s = 2.0 * np.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2])
        return (R[0,2]-R[2,0])/s, (R[0,1]+R[1,0])/s, 0.25*s, (R[1,2]+R[2,1])/s
    else:
        s = 2.0 * np.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1])
        return (R[1,0]-R[0,1])/s, (R[0,2]+R[2,0])/s, (R[1,2]+R[2,1])/s, 0.25*s

images_path = f"{COLMAP_DIR}/sparse/0/images.bin"
with open(images_path, 'wb') as f:
    f.write(struct.pack('<Q', len(frame_names)))
    for i, fname in enumerate(frame_names):
        if SLAM_TRAJ is not None and i < len(SLAM_TRAJ):
            # TUM format: timestamp x y z qx qy qz qw  (camera-to-world)
            _, tx, ty, tz, qx, qy, qz, qw = SLAM_TRAJ[i]
            R_c2w = quat_to_rot(qw, qx, qy, qz)
            t_c2w = np.array([tx, ty, tz])
            # Invert to world-to-camera (COLMAP convention)
            R_w2c = R_c2w.T
            t_w2c = -R_w2c @ t_c2w
            cqw, cqx, cqy, cqz = rot_to_quat(R_w2c)
            ctx, cty, ctz = t_w2c
        else:
            cqw, cqx, cqy, cqz = 1.0, 0.0, 0.0, 0.0
            ctx, cty, ctz = i * 0.1, 0.0, 0.0

        f.write(struct.pack('<i', i + 1))
        for v in [cqw, cqx, cqy, cqz, ctx, cty, ctz]:
            f.write(struct.pack('<d', v))
        f.write(struct.pack('<i', 1))          # camera_id
        f.write(fname.encode('utf-8') + b'\x00')
        f.write(struct.pack('<Q', 0))          # 0 keypoints

src = "MASt3R-SLAM" if SLAM_TRAJ is not None else "identity (fallback)"
print(f"✓ images.bin  {len(frame_names)} images, poses from: {src}")

In [ ]:
points_path = f"{COLMAP_DIR}/sparse/0/points3D.bin"

if SLAM_PLY and os.path.exists(SLAM_PLY):
    ply  = PlyData.read(SLAM_PLY)
    verts = ply['vertex']
    xyz  = np.column_stack([verts['x'], verts['y'], verts['z']])
    rgb  = np.column_stack([verts['red'], verts['green'], verts['blue']]).astype(np.uint8) \
           if 'red' in verts.data.dtype.names \
           else np.full((len(xyz), 3), 128, dtype=np.uint8)

    valid  = np.all(np.isfinite(xyz), axis=1)
    norms  = np.linalg.norm(xyz, axis=1)
    valid &= (norms > 1e-6) & (norms < 100.0)
    xyz, rgb = xyz[valid], rgb[valid]

    max_pts = 100_000
    if len(xyz) > max_pts:
        idx = np.random.choice(len(xyz), max_pts, replace=False)
        xyz, rgb = xyz[idx], rgb[idx]

    with open(points_path, 'wb') as f:
        f.write(struct.pack('<Q', len(xyz)))
        for j in range(len(xyz)):
            f.write(struct.pack('<Q', j + 1))
            for v in xyz[j]: f.write(struct.pack('<d', float(v)))
            for c in rgb[j]: f.write(struct.pack('<B', int(c)))
            f.write(struct.pack('<d', 0.0))   # error
            f.write(struct.pack('<Q', 0))     # empty track

    print(f"✓ points3D.bin  {len(xyz)} points")
else:
    with open(points_path, 'wb') as f:
        f.write(struct.pack('<Q', 0))
    print("✓ points3D.bin  0 points (no PLY — gsplat will use random init)")

print(f"\n✓ COLMAP workspace ready at {COLMAP_DIR}/")

---
## 4. Train 3D Gaussian Splatting (gsplat)

Fixes vs old notebook:
- `git clone` now pins `-b v1.3.0` (old: cloned latest main = API breakage)
- All 10 trainer deps now installed explicitly
- `examples/datasets/__init__.py` created (was missing → ModuleNotFoundError)
- `colmap.py` patched for `align_principal_axes` rename
- CUDA pre-compile step added (prevents JIT OOM on first training step)
- **NEW: `.pt` → `.ply` conversion cell** — gsplat saves checkpoints, not PLY files.
  The old notebook searched for a `.ply` that never existed.

Expected time: ~30–60 min on T4, ~15–30 min on A100.

In [ ]:
# gsplat pip package (the importable library)
!pip install -q gsplat==1.3.0

# FIX: all trainer script dependencies were missing in the old notebook
!pip install -q tyro viser imageio[ffmpeg] tensorboard \
    "torchmetrics[image]" opencv-python tqdm scipy nerfview \
    splines pycolmap PyYAML piexif

# FIX: pin to v1.3.0 so trainer scripts match the installed package
if not os.path.exists(GSPLAT_REPO):
    !git clone -b v1.3.0 https://github.com/nerfstudio-project/gsplat.git {GSPLAT_REPO}

    # FIX: __init__.py was missing → ModuleNotFoundError on first import
    !touch {GSPLAT_REPO}/examples/datasets/__init__.py

    # FIX: download known-working colmap.py + exif.py (function renames in main branch)
    !wget -q -O {GSPLAT_REPO}/examples/datasets/colmap.py \
        "https://github.com/nerfstudio-project/gsplat/raw/9b9f98a5b440531376b4a5386aea49f8e820203b/examples/datasets/colmap.py"
    !wget -q -O {GSPLAT_REPO}/examples/exif.py \
        "https://raw.githubusercontent.com/nerfstudio-project/gsplat/main/examples/exif.py"
    !wget -q -O {GSPLAT_REPO}/examples/datasets/exif.py \
        "https://raw.githubusercontent.com/nerfstudio-project/gsplat/main/examples/exif.py"

    # FIX: rename align_principal_axes → align_principle_axes (viser API change)
    !sed -i 's/align_principal_axes/align_principle_axes/g' \
        {GSPLAT_REPO}/examples/datasets/colmap.py

    print("✓ gsplat repo cloned and patched")
else:
    print("✓ gsplat repo already exists")

# FIX: pre-compile CUDA extensions now so the first training iteration
# doesn't JIT-compile and OOM the T4
print("Pre-compiling gsplat CUDA extensions (~5 min)...")
!rm -rf ~/.cache/torch_extensions/
!MAX_JOBS=2 python -c "import gsplat.cuda._backend; print('✓ CUDA compiled')"

In [ ]:
os.makedirs(GSPLAT_OUTPUT, exist_ok=True)

%cd {GSPLAT_REPO}

!MAX_JOBS=1 python examples/simple_trainer.py default \
    --data_dir {COLMAP_DIR} \
    --data_factor 1 \
    --result_dir {GSPLAT_OUTPUT} \
    --disable_viewer

print("\n✓ gsplat training complete")

In [ ]:
# FIX: THIS CELL WAS COMPLETELY MISSING FROM THE OLD NOTEBOOK.
#
# gsplat's simple_trainer.py saves checkpoints as .pt files, NOT .ply files.
# The old notebook searched for .ply files in the output dir, found nothing
# (or accidentally picked up the init point cloud), and FINAL_PLY was never
# set to the actual trained splat.
#
# This cell converts the final .pt checkpoint into standard 3DGS .ply format.
import torch
from plyfile import PlyData, PlyElement

# FIX: recursive=True was also missing in old glob call
ckpt_paths = sorted(glob.glob(f"{GSPLAT_OUTPUT}/**/ckpts/*.pt", recursive=True))

if not ckpt_paths:
    print("⚠ No gsplat checkpoints found in:", GSPLAT_OUTPUT)
    print("  Check training output above for errors.")
    FINAL_PLY = SLAM_PLY  # fall back to SLAM point cloud
    print(f"  Using SLAM point cloud as fallback: {FINAL_PLY}")
else:
    ckpt_path = ckpt_paths[-1]  # last checkpoint = highest step
    print(f"Loading checkpoint: {ckpt_path}")

    ckpt   = torch.load(ckpt_path, map_location="cpu")
    splats = ckpt["splats"]

    means     = splats["means"].numpy()        # (N, 3)
    scales    = splats["scales"].numpy()       # (N, 3) — log scale
    quats     = splats["quats"].numpy()        # (N, 4) — w x y z
    opacities = splats["opacities"].numpy()    # (N,) or (N,1) — raw logit
    sh0       = splats["sh0"].numpy()          # (N, 1, 3) — DC SH
    shN       = splats["shN"].numpy() if "shN" in splats else None  # (N, K, 3)

    N    = means.shape[0]
    sh0  = sh0.reshape(N, 3)
    if shN is not None:
        shN = shN.reshape(N, -1)   # (N, K*3)

    # Build PLY dtype (standard 3DGS format)
    dtype = [
        ('x','f4'),('y','f4'),('z','f4'),
        ('nx','f4'),('ny','f4'),('nz','f4'),
        ('f_dc_0','f4'),('f_dc_1','f4'),('f_dc_2','f4'),
    ]
    for i in range(45):
        dtype.append((f'f_rest_{i}', 'f4'))
    dtype += [
        ('opacity','f4'),
        ('scale_0','f4'),('scale_1','f4'),('scale_2','f4'),
        ('rot_0','f4'),('rot_1','f4'),('rot_2','f4'),('rot_3','f4'),
    ]

    el = np.empty(N, dtype=dtype)
    el['x'], el['y'], el['z'] = means[:,0], means[:,1], means[:,2]
    el['nx'] = el['ny'] = el['nz'] = 0
    el['f_dc_0'], el['f_dc_1'], el['f_dc_2'] = sh0[:,0], sh0[:,1], sh0[:,2]

    if shN is not None:
        for i in range(min(45, shN.shape[1])):
            el[f'f_rest_{i}'] = shN[:, i]
    else:
        for i in range(45):
            el[f'f_rest_{i}'] = 0

    el['opacity']  = opacities.flatten()
    el['scale_0']  = scales[:,0]
    el['scale_1']  = scales[:,1]
    el['scale_2']  = scales[:,2]
    el['rot_0']    = quats[:,0]   # w
    el['rot_1']    = quats[:,1]   # x
    el['rot_2']    = quats[:,2]   # y
    el['rot_3']    = quats[:,3]   # z

    FINAL_PLY = f"{GSPLAT_OUTPUT}/splat_final.ply"
    PlyData([PlyElement.describe(el, 'vertex')]).write(FINAL_PLY)

    size_mb = os.path.getsize(FINAL_PLY) / 1e6
    print(f"\n✓ Exported {N:,} Gaussians → {FINAL_PLY} ({size_mb:.1f} MB)")

---
## 5. Grounded-SAM-2 Semantic Masks

Fix vs old notebook:
- Old code used `build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", ckpt)` with a
  relative path that doesn't exist after pip install → `FileNotFoundError`.
- New code uses `SAM2ImagePredictor.from_pretrained("facebook/sam2.1-hiera-large")`
  which resolves both checkpoint and config from HuggingFace Hub automatically.
  The manual checkpoint download is no longer needed.

Expected time: ~5–15 min for 100 frames on T4.

In [ ]:
!pip install -q sam2 supervision
print("✓ sam2 + supervision installed")

In [ ]:
from tqdm import tqdm
from PIL import Image as PILImage

os.makedirs(MASKS_DIR, exist_ok=True)

TEXT_PROMPT = "chair. table. sofa. monitor. laptop. cup. floor. wall. door. shelf. bed. lamp."

frame_paths = sorted([
    os.path.join(FRAMES_DIR, f)
    for f in os.listdir(FRAMES_DIR)
    if f.endswith(('.jpg', '.png'))
])
print(f"Frames        : {len(frame_paths)}")
print(f"Text prompt   : {TEXT_PROMPT}")

try:
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    from sam2.sam2_image_predictor import SAM2ImagePredictor

    # --- Grounding DINO ---
    gdino_id  = "IDEA-Research/grounding-dino-tiny"
    gdino_proc  = AutoProcessor.from_pretrained(gdino_id)
    gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(gdino_id).to("cuda")
    print("✓ Grounding DINO loaded")

    # FIX: old code used build_sam2(relative_config_path, ckpt) which raises
    # FileNotFoundError when SAM2 is installed as a pip package (config
    # lives inside the package, not in the working directory).
    # from_pretrained resolves everything automatically via HuggingFace Hub.
    sam2_predictor = SAM2ImagePredictor.from_pretrained(
        "facebook/sam2.1-hiera-large", device="cuda"
    )
    print("✓ SAM 2.1 loaded")

    all_masks_info = []

    for i, frame_path in enumerate(tqdm(frame_paths)):
        image     = cv2.imread(frame_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w      = image.shape[:2]
        pil_img   = PILImage.fromarray(image_rgb)

        # Detect with Grounding DINO
        inputs = gdino_proc(
            images=pil_img, text=TEXT_PROMPT, return_tensors="pt"
        ).to("cuda")
        with torch.no_grad():
            outputs = gdino_model(**inputs)
        results = gdino_proc.post_process_grounded_object_detection(
            outputs, inputs["input_ids"],
            threshold=0.3, text_threshold=0.25,
            target_sizes=[(h, w)],
        )

        boxes  = results[0]["boxes"].cpu().numpy()
        scores = results[0]["scores"].cpu().numpy()
        labels = results[0]["labels"]

        frame_masks = []
        frame_stem  = os.path.basename(frame_path).rsplit('.', 1)[0]

        if len(boxes) > 0:
            sam2_predictor.set_image(image_rgb)
            masks, _, _ = sam2_predictor.predict(
                box=boxes, multimask_output=False
            )
            for j in range(len(masks)):
                mask_img  = masks[j].squeeze().astype(np.uint8) * 255
                mask_file = f"{frame_stem}_mask_{j:03d}.png"
                cv2.imwrite(os.path.join(MASKS_DIR, mask_file), mask_img)
                frame_masks.append({
                    "mask_file"  : mask_file,
                    "label"      : labels[j] if j < len(labels) else "unknown",
                    "confidence" : float(scores[j]) if j < len(scores) else 0.0,
                    "instance_id": j,
                })

        all_masks_info.append({
            "frame"      : os.path.basename(frame_path),
            "frame_index": i,
            "masks"      : frame_masks,
        })

    with open(os.path.join(MASKS_DIR, "masks.json"), 'w') as f:
        json.dump(all_masks_info, f, indent=2)

    total = sum(len(x["masks"]) for x in all_masks_info)
    print(f"\n✓ Saved {total} masks across {len(frame_paths)} frames → {MASKS_DIR}/")

except Exception as e:
    import traceback
    print(f"\n⚠ Segmentation error: {e}")
    traceback.print_exc()
    print("Continuing — semantic masks can be added later.")

---
## 6. Save Everything to Google Drive

In [ ]:
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# 1. COLMAP workspace
colmap_dst = f"{DRIVE_OUTPUT}/colmap"
if os.path.exists(colmap_dst): shutil.rmtree(colmap_dst)
shutil.copytree(COLMAP_DIR, colmap_dst)
print(f"✓ COLMAP workspace → {colmap_dst}/")

# 2. Gaussian splat .ply
if FINAL_PLY and os.path.exists(FINAL_PLY):
    ply_dst = f"{DRIVE_OUTPUT}/splat.ply"
    shutil.copy2(FINAL_PLY, ply_dst)
    print(f"✓ Gaussian splat ({os.path.getsize(ply_dst)/1e6:.0f} MB) → {ply_dst}")
else:
    print("⚠ No final .ply — skipping")

# 3. Semantic masks
if os.path.exists(MASKS_DIR) and os.listdir(MASKS_DIR):
    masks_dst = f"{DRIVE_OUTPUT}/masks"
    if os.path.exists(masks_dst): shutil.rmtree(masks_dst)
    shutil.copytree(MASKS_DIR, masks_dst)
    n_masks = len([f for f in os.listdir(MASKS_DIR) if f.endswith('.png')])
    print(f"✓ Masks ({n_masks} files) → {masks_dst}/")

# 4. Raw SLAM outputs
if SLAM_PLY and os.path.exists(SLAM_PLY):
    shutil.copy2(SLAM_PLY, f"{DRIVE_OUTPUT}/slam_pointcloud.ply")
    print(f"✓ SLAM point cloud → {DRIVE_OUTPUT}/slam_pointcloud.ply")

if SLAM_TRAJ is not None:
    np.savetxt(f"{DRIVE_OUTPUT}/slam_trajectory.txt", SLAM_TRAJ, fmt='%.6f')
    print(f"✓ SLAM trajectory → {DRIVE_OUTPUT}/slam_trajectory.txt")

# 5. SLAM logs (for debugging)
if os.path.exists(SLAM_LOGS):
    logs_dst = f"{DRIVE_OUTPUT}/slam_logs"
    if os.path.exists(logs_dst): shutil.rmtree(logs_dst)
    shutil.copytree(SLAM_LOGS, logs_dst)
    print(f"✓ SLAM logs → {logs_dst}/")

print(f"""
{'='*60}
 ALL RESULTS SAVED
{'='*60}
 Location: {DRIVE_OUTPUT}/

 On your Mac, download and place as:
   colmap/     → data/{SCENE_NAME}/colmap/
   splat.ply   → outputs/{SCENE_NAME}/splat.ply
   masks/      → data/{SCENE_NAME}/masks/

 Then run:
   bash run.sh assets/videos/my_scene.mp4 {SCENE_NAME}
{'='*60}
""")